<a href="https://colab.research.google.com/github/nabilah-afrin/recommendation_system_rokomri_books/blob/secondary/notebooks/Categories_bn_books.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# %cd /content/drive/MyDrive/Rokomari Recommendation Dataset
%cd /content/drive/MyDrive/Dataset/rokomari_books/Rokomari Recommendation Dataset/Datasets

/content/drive/.shortcut-targets-by-id/1SdeIcOv6xY-c8y2UcMwPLYx_BaB0zdXx/Rokomari Recommendation Dataset/Datasets


In [3]:
!ls

 corrected_language.csv        rokomari_book_data_v2.csv	  'scraping log.txt'
'Data Analysis Report.gdoc'    rokomari_books_only_bangla_v2.csv   wrong_language_url.txt
 mixed_title_rokomari_v2.csv   rokomari_v2.csv
 rokomari_book_data.csv        rokomari_v2.ipynb


In [4]:
# !pip install googletrans==4.0.0-rc1
# !pip install googletrans==4.0.0rc1 --quiet

In [5]:
!pip install langdetect -q -q -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 10.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [6]:
# !pip install -U easynmt

In [7]:
!pip install deep_translator --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.7 MB/s eta 0:00:00


In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import langdetect as ld
# import googletrans as gt
import deep_translator as dt
from deep_translator import GoogleTranslator

# Only Bangla Books

In [9]:
# load the rokomari_bn_books.csv
df_bn = pd.read_csv("rokomari_books_only_bangla_v2.csv")

# comment mine and uncomment yours
# df_bn = pd.read_csv("/content/drive/MyDrive/Rokomari Recommendation Dataset/df_bn.csv")

In [10]:
df_bn.head(5)

,book_id,title,author,publisher,publisher_name_english,categories,category_english,edition,isbn,summary,...,n_reviews,price,offer_price,book_url,prod_img_link,availability,product_category,actual_rating,bangla_title,language_title
0,350167,অমানুষিক,মনোরঞ্জন ব্যাপারী,একা,Eka (India),পশ্চিমবঙ্গের বই,West Bengal Books,6 March 2023,9789357762298,No summary,...,0,450.0,450.0,https://www.rokomari.com/book/350167/amanushik,https://img.cf.rokomari.com/ProductNew20190903...,Request For Reprint,book,0.0,অমানুষিক,bn
1,377700,নির্বাচিত গল্প সংকলন,লু স্যুন,ছাড়পত্র প্রকাশন (ইন্ডিয়া),Charpatra Prakashan (India),পশ্চিমবঙ্গের বই: সমকালীন গল্প,West Bengal Books: Contemporary Story,Edition,9788194097303,No summary,...,0,320.0,320.0,https://www.rokomari.com/book/377700/nirbachit...,https://img.cf.rokomari.com/ProductNew20190903...,Request For Reprint,book,0.0,নির্বাচিত গল্প সংকলন,bn
2,377718,তিমুর ও তার দলবল,আর্কাদি গাইদার,ছাড়পত্র প্রকাশন (ইন্ডিয়া),Charpatra Prakashan (India),পশ্চিমবঙ্গের বই: শিশু-কিশোর উপন্যাস,West Bengal Books: Children and Teens Novel,Edition,9788193860939,No summary,...,0,180.0,180.0,https://www.rokomari.com/book/377718/timur-o-t...,https://img.cf.rokomari.com/ProductNew20190903...,Request For Reprint,book,0.0,তিমুর ও তার দলবল,bn
3,189529,চেক ডিসঅনার মামলার সহজ ভাষ্য,মোঃ কাইছার হামিদ,এ.কে লিগ্যাল সল্যুশন,A.K Legal Solution,ব্যাংকিং এন্ড কমার্স ল,Banking and Commerce Law,1st Published,No ISBN,* চেক ডিসঅনার ও মামলা দায়েরের পদ্ধতি সংক্রান্...,...,14,250.0,175.0,https://www.rokomari.com/book/189529/cheque-di...,https://img.cf.rokomari.com/ProductNew20190903...,Not Available,book,5.0,চেক ডিসঅনার মামলার সহজ ভাষ্য,bn
4,325781,হাইকোর্ট এক্সাম ফর্মুলা,মোঃ কাইছার হামিদ,A.K Legal Solution,A.K Legal Solution,Advocacy/ Adjudication Law,Advocacy/ Adjudication Law,Edition,9789843545589,"""High Court Exam Formula"" book will be very he...",...,0,750.0,500.0,https://www.rokomari.com/book/325781/high-cour...,https://img.cf.rokomari.com/ProductNew20190903...,Request For Reprint,book,0.0,হাইকোর্ট এক্সাম ফর্মুলা,bn


# Useful Functions

In [11]:
import re
import time
# from easynmt import EasyNMT
# model = EasyNMT('m2m_100_1.2B')
# print(model.translate('Bank Job Entrance Exam Preparation', target_lang='bn'))

def detect_language(row):
    try:
        lang = ld.detect(str(row))
    except:
        return 'unknown'
    return lang

def keep_only_bangla(row):
    """
    The unicode range between \u0980 - \u09FF defines the Bangla characters
    and digits in the Unicode character set
    """
    return re.sub(r'[^-|,|\u0980-\u09FF| ]+', '', str(row)).strip()

def translate_author_names(row):
    text = str(row)
    # time.sleep(8)
    translator = gt.Translator()
    translation = translator.translate(text=text, dest='bn')

    return translation.text

def translate_to_bangla(row):
    return GoogleTranslator(source='auto', target='bn').translate(text=str(row))

def remove_unnecessary_characters(row):
    # pattern = r'[ও|():/, ]+'
    digit_map = str.maketrans('0123456789', '০১২৩৪৫৬৭৮৯')
    text = re.sub(r'[:|ঃ]', ",", str(row))
    # text = re.sub(r'[/]', ", ", text)
    text = re.sub(r'[(|)]', "", text)
    text = re.sub(r'(\s)ও(\s)|(,\s)ও(\s)|(\s)এবং(\s)|/', ', ', text)
    # text = re.sub(r'\sও\s', ', ', text)
    text = re.sub(r'[\d|\d+]', lambda x: x.group(0).translate(digit_map), text)

    return text.strip()

def fix_more_categories_issue(row):
    text = re.sub(r'ক্লাস ৯, ১০', "ক্লাস ৯-১০,", str(row))
    text = re.sub(r"(পাঠ\s)সোহায়িকা|(পাঠো\s)সোহায়িকা|(পাঠের\s)সোহায়িকা", "পাঠ্য সহায়িকা", text)

    return text


In [12]:
# pattern = r'(\s)ও(\s)|(,\s)ও(\s)|(\s)এবং(\s)|/'
text = "শিশু-কিশোরঃ বই এবং (উপন্যাস) ও, কিতাব/পুস্তিকা 0-10, ক্লাস- 1, বইমেলা 2024, 9, 10"
digit_map = str.maketrans('0123456789', '০১২৩৪৫৬৭৮৯')
# # text = "রহস্য, গোয়েন্দা, ভৌতিক, মিথ, থ্রিলার, ও অ্যাডভেঞ্চার: অনুবাদ ও ইংরেজি"
# text = re.sub(r'[:|ঃ]', ",", str(text))
# text = re.sub(r'[/]', ", ", text)
# text = re.sub(pattern, ", ", text)
# text = re.sub(r'[(|)]', "", text)
# text = re.sub(r'(\d+)-(\d+)', lambda x: x.group(0).translate(digit_map), text)
text = re.sub(r'[\d|\d+]', lambda x: x.group(0).translate(digit_map), text)
print(text)
# # text = re.sub(r'(,+)', "", text)
# print(text)

শিশু-কিশোরঃ বই এবং (উপন্যাস) ও, কিতাব/পুস্তিকা ০-১০, ক্লাস- ১, বইমেলা ২০২৪, ৯, ১০


In [13]:
# text = "শিশু-কিশোর: বই এবং (উপন্যাস) ও কিতাব/পুস্তিকা"
# text = "এসএসসি ১ম বর্ষ"
# text = "যখন ৪-৮: গল্প"
# text = re.sub(r'[:]', ",", str(text))
# text = re.sub(r'[/]', ", ", text)
# text = re.sub(r'[(|)]', "", text)
# text = re.sub(r'\sএবং\s', ', ', text)
# text = re.sub(r'\sও\s', ', ', text)
# # text = re.sub(r' ', "", text)
# print(text)


# 2. Categories


In [14]:
df_bn['categories'].nunique()

2163

In [15]:
df_bn['categories'].value_counts().reset_index().loc[:20]

,categories,count
0,বাংলা কবিতা,21039
1,সমকালীন উপন্যাস,12184
2,পশ্চিমবঙ্গের বই,10466
3,সমকালীন গল্প,6794
4,শিশু-কিশোর গল্প,5074
5,ছড়া,2566
6,চিরায়ত উপন্যাস,2411
7,রোমান্টিক কবিতা,2180
8,বিবিধ বিষয়ক প্রবন্ধ,1918
9,পশ্চিমবঙ্গের বই: প্রবন্ধ,1765


In [16]:
df_bn.loc[df_bn['categories'] == "Hadith & Sunnah"]

,book_id,title,author,publisher,publisher_name_english,categories,category_english,edition,isbn,summary,...,n_reviews,price,offer_price,book_url,prod_img_link,availability,product_category,actual_rating,bangla_title,language_title
5212,221841,কাশফুল বারী আম্মা ফি সহিহিল বুখারি ভলিউম ২০ খণ...,শায়খুল হাদিস আল্লামা সালিমুল্লাহ খান র.,Madani Kutubkhana,Madani Kutubkhana,Hadith & Sunnah,Hadith & Sunnah,1st edition,No ISBN,No summary,...,0,750.0,510.0,https://www.rokomari.com/book/221841/kashful-b...,https://img.cf.rokomari.com/ProductNew20190903...,Add to Cart,book,0.00,কাশফুল বারী আম্মা ফি সহিহিল বুখারি ভলিউম ২০ খণ...,bn
5480,244729,আছারুল হাদীস,মাকতাবাতুল হিজায,Maktabatul Hijaz,Maktabatul Hijaz,Hadith & Sunnah,Hadith & Sunnah,No Edition,No ISBN,No summary,...,0,550.0,495.0,https://www.rokomari.com/book/244729/acharul-h...,https://img.cf.rokomari.com/ProductNew20190903...,Add to Cart,book,0.00,আছারুল হাদীস,bn
12475,116623,কুরআনের আলো রোযার হাদীস,মাওলানা মুহাম্মাদ সিরাজুল ইসলাম,Siddikia Publications,Siddikia Publications,Hadith & Sunnah,Hadith & Sunnah,1st Published,9848913629,No summary,...,0,700.0,455.0,https://www.rokomari.com/book/116623/quraner-a...,https://img.cf.rokomari.com/ProductNew20190903...,Add to Cart,book,0.00,কুরআনের আলো রোযার হাদীস,bn
14050,427343,রাসূলুল্লাহ (সাঃ) এর সলাত এবং আক্বীদাহ ও জরুরি...,আল্লামা আবূ মুহাম্মাদ ‘আলীমুদ্দীন (রহ.),Sunan prokasoni,Sunan prokasoni,Hadith & Sunnah,Hadith & Sunnah,3rd Published,No ISBN,জিবরীল (আ) ১০ ওয়াক্ত সলাত পড়ে সলাতের সঠিক পদ্ধ...,...,0,500.0,400.0,https://www.rokomari.com/book/427343/rasululla...,https://img.cf.rokomari.com/ProductNew20190903...,Add to Cart,book,0.00,রাসূলুল্লাহ সাঃ এর সলাত এবং আক্বীদাহ ও জরুরি ম...,bn
17448,428703,সহিহ হাদিসে কুদসী,কামারুজ্জামান বিন আব্দুল মালেক আল-শিবলী আল-আযহারী,Trinolota Prokash,Trinolota Prokash,Hadith & Sunnah,Hadith & Sunnah,1st Published,978984551040,‘সহিহ হাদিসে কুদসী’ হাদিসের নানা বিষয় নিয়ে রচি...,...,0,300.0,270.0,https://www.rokomari.com/book/428703/sahih-had...,https://img.cf.rokomari.com/ProductNew20190903...,Add to Cart,book,0.00,সহিহ হাদিসে কুদসী,bn
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
203365,423351,মূলনীতিপূর্ণ ১০০ সহীহ হাদীস,মোঃ মাহবুবুর রহমান বিন মুসলেহুদ্দীন,Wahidiya Islamiya Library,Wahidiya Islamiya Library,Hadith & Sunnah,Hadith & Sunnah,1st Published,No ISBN,"ইসলামী শরীয়তের দ্বিতীয় মূল উৎস হল, হাদীস যা কু...",...,0,75.0,58.0,https://www.rokomari.com/book/423351/mulnitipu...,https://img.cf.rokomari.com/ProductNew20190903...,Add to Cart,book,0.00,মূলনীতিপূর্ণ ১০০ সহীহ হাদীস,bn
203380,223656,সহীহ আত-তারগীব ওয়াত তারহীব - ২য় খণ্ড,আল্লামা মুহাম্মদ নাসীরুদ্দীন আলবানী (রহঃ),Wahidiya Islamiya Library,Wahidiya Islamiya Library,Hadith & Sunnah,Hadith & Sunnah,1st Published,No ISBN,সদ্য প্রকাশিত নতুন বই । #সহীহ_আত_তারগীব_ওয়াত_ত...,...,0,750.0,578.0,https://www.rokomari.com/book/223656/sahih-at-...,https://img.cf.rokomari.com/ProductNew20190903...,Add to Cart,book,0.00,সহীহ আততারগীব ওয়াত তারহীব ২য় খণ্ড,bn
203433,407829,হারানো সুন্নাহ,শাইখ ড. মুতলাক বিন জাসির বিন মুতলাক বিন ফারিস ...,Wahidiya Islamiya Library,Wahidiya Islamiya Library,Hadith & Sunnah,Hadith & Sunnah,1st Published,No ISBN,"মানুষ যেসব সুন্নাহর ওপর আমল করা ছেড়ে দিয়েছে, য...",...,1,125.0,96.0,https://www.rokomari.com/book/407829/harano-su...,https://img.cf.rokomari.com/ProductNew20190903...,Add to Cart,book,5.00,হারানো সুন্নাহ,bn
203446,257543,হাদীস-শাস্ত্রের পারিভাষিক জ্ঞান,শাইখ আব্দুল হামীদ আল-ফাইযী আল-মাদানী,Wahidiya Islamiya Library,Wahidiya Islamiya Library,Hadith & Sunnah,Hadith & Sunnah,1st Published,No ISBN,ইসলামী বই পড়তে গিয়ে আমরা প্রায়শই কিছু পারিভ...,...,0,163.0,126.0,https://www.rokomari.com/book/257543/hadis-sas...,https://img.cf.rokomari.com/ProductNew20190903...,Add to Cart,book,0.00,হাদীসশাস্ত্রের পারিভাষিক জ্ঞান,bn


## 2.1 Language variations in Categories

In [17]:
df_bn['language_categories'] = df_bn['categories'].apply(detect_language)

In [18]:
df_bn['language_categories'].value_counts()

,count
language_categories,
bn,200013
en,2467
id,528
de,356
so,224
af,218
fr,145
tl,112
nl,104


In [19]:
df_bn.loc[df_bn['language_categories'] == 'ar']

,book_id,title,author,publisher,publisher_name_english,categories,category_english,edition,isbn,summary,...,price,offer_price,book_url,prod_img_link,availability,product_category,actual_rating,bangla_title,language_title,language_categories
68390,381070,কুরআনিক গ্রামার,উম্মে কুলসুম মুনমুন,ঐতিহ্য,Oitijjhya,كتب اللغة العربية والأردية (আরবি ও উর্দূ ভাষার...,Arabic & Urdu Languages Books,1st Published,9789847762524,লেখক গত দুই বছর ধরে স্ব-উদ্যোগে আল কুরআনরে ভাষ...,...,500.0,430.0,https://www.rokomari.com/book/381070/quranik-g...,https://img.cf.rokomari.com/ProductNew20190903...,Add to Cart,book,0.0,কুরআনিক গ্রামার,bn,ar
91748,221591,আল আদাবুল আরবী ইনদা আলী তানতাভী ১-৩ খণ্ড,ড. শায়খ আলী তানতাবী রাহ.,মাকতাবাতুল আফনান,Maktabatul Afnan,كتب اللغة العربية والأردية (আরবি ও উর্দূ ভাষার...,Arabic & Urdu Languages Books,1st published,No ISBN,আলী তান্তাভি রচিত আরবি ভাষায় বিভিন্ন প্রবন্ধ য...,...,1000.0,1000.0,https://www.rokomari.com/book/221591/al-adabul...,https://img.cf.rokomari.com/ProductNew20190903...,Request For Reprint,book,0.0,আল আদাবুল আরবী ইনদা আলী তানতাভী ১৩ খণ্ড,bn,ar
107459,389931,আল-আযকার,ইমাম আবু যাকারিয়া ইয়াহইয়া বিন শরফ আন-নববী রহ.,জোনাকী প্রকাশনী,Jonaki Prokashoni,كتب اللغة العربية والأردية (আরবি ও উর্দূ ভাষার...,Arabic & Urdu Languages Books,1st Published,No ISBN,সকল প্রশংসা মহান আল্লাহ রাব্বুল আলামিনের জন্য।...,...,900.0,746.0,https://www.rokomari.com/book/389931/al-azkar,https://img.cf.rokomari.com/ProductNew20190903...,Add to Cart,book,0.0,আলআযকার,bn,ar
108033,387977,বুখারী শরীফ (১ - ১০ম খন্ড),জননী বুক ডিপো,জননী বুক ডিপো,Jononi Book Depo,كتب اللغة العربية والأردية (আরবি ও উর্দূ ভাষার...,Arabic & Urdu Languages Books,1st Published,No ISBN,No summary,...,3800.0,1786.0,https://www.rokomari.com/book/387977/bukhari-s...,https://img.cf.rokomari.com/ProductNew20190903...,Add to Cart,book,0.0,বুখারী শরীফ ১ ১০ম খন্ড,bn,ar
111424,393172,আল্লাহ ওয়ালাদের মকবূল মুনাজাত ও তারানা,মাও. ইবরাহীম সেনবাগী,ইত্তিহাদ পাবলিকেশন,Ittihad Publication,كتب اللغة العربية والأردية (আরবি ও উর্দূ ভাষার...,Arabic & Urdu Languages Books,1st Published,No ISBN,মহান রব্বুল আলামীন বৈচিত্রময় এ বসুন্ধরায় যুগে ...,...,120.0,60.0,https://www.rokomari.com/book/393172/allah-wal...,https://img.cf.rokomari.com/ProductNew20190903...,Add to Cart,book,0.0,আল্লাহ ওয়ালাদের মকবূল মুনাজাত ও তারানা,bn,ar
117017,250650,তারাশে (উর্দূ গল্প),হাকীমুল উম্মত প্রকাশনী,حكيم الامت پركاشني (হাকীমুল উম্মত প্রকাশনী),Hakimul Ummot Prokashoni,كتب اللغة العربية والأردية (আরবি ও উর্দূ ভাষার...,Arabic & Urdu Languages Books,No Edition,No ISBN,No summary,...,200.0,200.0,https://www.rokomari.com/book/250650/tarashe-u...,https://img.cf.rokomari.com/ProductNew20190903...,Add to Cart,book,0.0,তারাশে উর্দূ গল্প,bn,ar
119831,247636,তাইসীরুল উসুল,মাওলানা মুহাম্মদ নাঈম (হাফিজাহুল্লাহ),গ্রন্থালয়,Gronthaloy,كتب اللغة العربية والأردية (আরবি ও উর্দূ ভাষার...,Arabic & Urdu Languages Books,No Edition,No ISBN,No summary,...,400.0,220.0,https://www.rokomari.com/book/247636/taisirul-...,https://img.cf.rokomari.com/ProductNew20190903...,Add to Cart,book,0.0,তাইসীরুল উসুল,bn,ar
140063,381526,যাকাত ডায়েরি,শায়খ আব্দুল জাব্বার জাহাঙ্গীর,দারুল হিকমাহ পাবলিকেশন্স লিমিটেড,Darul Hikmah Publications Ltd.,كتب اللغة العربية والأردية (আরবি ও উর্দূ ভাষার...,Arabic & Urdu Languages Books,1st Published,9789848063019,যাকাত সম্পদশালীদের জন্য একটি ফরয ইবাদাত। এটি ই...,...,800.0,688.0,https://www.rokomari.com/book/381526/zakat-diary,https://img.cf.rokomari.com/ProductNew20190903...,Add to Cart,book,0.0,যাকাত ডায়েরি,bn,ar


In [21]:
df_bn.loc[df_bn['language_categories'] == 'en']

,book_id,title,author,publisher,publisher_name_english,categories,category_english,edition,isbn,summary,...,price,offer_price,book_url,prod_img_link,availability,product_category,actual_rating,bangla_title,language_title,language_categories
4,325781,হাইকোর্ট এক্সাম ফর্মুলা,মোঃ কাইছার হামিদ,A.K Legal Solution,A.K Legal Solution,Advocacy/ Adjudication Law,Advocacy/ Adjudication Law,Edition,9789843545589,"""High Court Exam Formula"" book will be very he...",...,750.0,500.0,https://www.rokomari.com/book/325781/high-cour...,https://img.cf.rokomari.com/ProductNew20190903...,Request For Reprint,book,0.0,হাইকোর্ট এক্সাম ফর্মুলা,bn,en
7,313765,ফিরে দেখা ১৯৭১ এর চিঠি ও কথা,ডঃ এ.কে.এম.এ. কাদের,A-Z Foundation,A-Z Foundation,"Diary, Letters and Memories","Diary, Letters and Memories",1st Edition,9789843542939,"The letters, clippings, scribbles, and documen...",...,850.0,731.0,https://www.rokomari.com/book/313765/reminisce...,https://img.cf.rokomari.com/ProductNew20190903...,Add to Cart,book,5.0,ফিরে দেখা ১৯৭১ এর চিঠি ও কথা,bn,en
23,113881,ঈশান ও তার সুপারহিরো বন্ধুরা,এ. এন. এম নূরুল হক,Adorn Books For Children (ABC),Adorn Books For Children (ABC),Story: Children and Teens,Story: Children and Teens,1st Published,9789842010637,No summary,...,100.0,86.0,https://www.rokomari.com/book/113881/eshan-and...,https://img.cf.rokomari.com/ProductNew20190903...,Add to Cart,book,5.0,ঈশান ও তার সুপারহিরো বন্ধুরা,bn,en
262,84950,বেহেশ্‌তী জেওর (১ হতে ১১ খণ্ড একত্রে),حكيم الامت مولانا اشرف علي تهانوي رح ( হাকীমুল...,Alif Publications,Alif Publications,Islamic Rules and Regulation and Provisions,Islamic Rules and Regulation and Provisions,1st Published,No ISBN,No Summary,...,450.0,338.0,https://www.rokomari.com/book/84950/beheshti-j...,https://img.cf.rokomari.com/ProductNew20190903...,Request For Reprint,book,0.0,বেহেশ্তী জেওর ১ হতে ১১ খণ্ড একত্রে,bn,en
264,85017,মোকছুদুল মোমীনীন বা বেহেশতের পুঞ্জী (সকল খণ্ড ...,মাওলানা খাজা আশরাফ আলী সাহেব,Alif Publications,Alif Publications,Islamic Practices and Guides,Islamic Practices and Guides,1st Published,No ISBN,No Summary,...,500.0,310.0,https://www.rokomari.com/book/85017/moksudul-m...,https://img.cf.rokomari.com/ProductNew20190903...,Request For Reprint,book,1.0,মোকছুদুল মোমীনীন বা বেহেশতের পুঞ্জী সকল খণ্ড এ...,bn,en
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
203606,188385,চারিত্রিক সৌন্দর্য কোন পথে?,ছানা উল্লাহ রিয়াদ,আল-ইমাম ফাউন্ডেশন,আল-ইমাম ফাউন্ডেশন,Ideal Lifestyle for Family and Personal,Ideal Lifestyle for Family and Personal,1st Published,No ISBN,No summary,...,60.0,42.0,https://www.rokomari.com/book/188385/charitrik...,https://img.cf.rokomari.com/ProductNew20190903...,Not Available,book,0.0,চারিত্রিক সৌন্দর্য কোন পথে?,bn,en
203607,188386,কুরআন ও আধুনিক বিজ্ঞান,আল্লামা ওবায়দুল্লাহ হামযাহ্,আল-ইমাম ফাউন্ডেশন,আল-ইমাম ফাউন্ডেশন,Islam and Science,Islam and Science,No Edition,No ISBN,No summary,...,40.0,28.0,https://www.rokomari.com/book/188386/quran-o-a...,https://img.cf.rokomari.com/ProductNew20190903...,Not Available,book,0.0,কুরআন ও আধুনিক বিজ্ঞান,bn,en
203608,188384,এসো হে মুমিন জান্নাতের পথে,আল্লামা ওবায়দুল্লাহ হামযাহ্,আল-ইমাম ফাউন্ডেশন,আল-ইমাম ফাউন্ডেশন,Islamic Practices and Guides,Islamic Practices and Guides,1st Published,No ISBN,No summary,...,60.0,42.0,https://www.rokomari.com/book/188384/eso-he-mu...,https://img.cf.rokomari.com/ProductNew20190903...,Not Available,book,0.0,এসো হে মুমিন জান্নাতের পথে,bn,en
204166,432557,"এডমিশন পকেট বুক, ইংলিশ গ্রামার ও লিটারেচার",মোঃ রোকনুজ্জামান সোহেল,Young Bengal Publications,Young Bengal Publications,University Admission Preparation,University Admission Preparation,1st Published,No ISBN,"ঢাবি, রাবি, চবি, জাবি, গুচ্ছ , Ari ( Cluster) ...",...,180.0,180.0,https://www.rokomari.com/book/432557/admission...,https://img.cf.rokomari.com/ProductNew20190903...,Request For Reprint,book,0.0,এডমিশন পকেট বুক ইংলিশ গ্রামার ও লিটারেচার,bn,en


## 2.2 Fixing categories detected as other language to Bangla and mixed lanugage

### 2.2.1 Fixing categories detected as other language to Bangla

In [22]:
for language in df_bn['language_categories'].value_counts().index:
    if language == 'bn' or language == 'ar':
        continue
    else:
        print(f"Correcting Language: {language}")
        filt = (df_bn['language_categories'] == language, 'categories')
        df_bn.loc[filt] = df_bn.loc[filt].apply(translate_to_bangla)

print("\nDone")

Correcting Language: en
Correcting Language: id
Correcting Language: de
Correcting Language: so
Correcting Language: af
Correcting Language: fr
Correcting Language: tl
Correcting Language: nl
Correcting Language: sw
Correcting Language: ro
Correcting Language: fi
Correcting Language: tr
Correcting Language: es
Correcting Language: it
Correcting Language: et
Correcting Language: hr
Correcting Language: ca
Correcting Language: lt
Correcting Language: cy
Correcting Language: sq
Correcting Language: vi
Correcting Language: pt
Correcting Language: sv
Correcting Language: no

Done


### 2.2.2 Fixing categories with mixed languages with Bangla

In [23]:
filt = (df_bn['language_categories'] == 'ar', 'categories')

In [24]:
df_bn.loc[filt] = df_bn.loc[filt].apply(keep_only_bangla)

In [25]:
df_bn.loc[filt]

,categories
68390,আরবি ও উর্দূ ভাষার বই
91748,আরবি ও উর্দূ ভাষার বই
107459,আরবি ও উর্দূ ভাষার বই
108033,আরবি ও উর্দূ ভাষার বই
111424,আরবি ও উর্দূ ভাষার বই
117017,আরবি ও উর্দূ ভাষার বই
119831,আরবি ও উর্দূ ভাষার বই
140063,আরবি ও উর্দূ ভাষার বই


In [29]:
df_bn.loc[df_bn['language_categories'] == 'en', 'categories']

,categories
71226,"উক্তি, Aphorisms & proverbs"
125454,"উক্তি, Aphorisms & proverbs"
148248,"উক্তি, Aphorisms & proverbs"


In [33]:
df_bn['language_categories'] = df_bn['categories'].apply(detect_language)
df_bn['language_categories'].value_counts()

,count
language_categories,
bn,204572


In [ ]:
# save the modified dataframe

df_bn.to_csv("rokomari_books_categories_corrected.csv", index=False)

## 3 Load the categories corrected dataframe from here

In [ ]:
df_bn = pd.read_csv("rokomari_books_categories_corrected.csv")

### 3.1 Use the Language detection model

In [ ]:
# df_bn['language_categories'] = df_bn['categories'].apply(detect_language)

In [ ]:
df_bn['language_categories'].value_counts()

,count
language_categories,
bn,98254


In [35]:
df_bn.loc[df_bn['categories'].str.contains('নবী-রাসূল'), 'categories']

,categories
5270,"নবী-রাসূল, সাহাবা, তাবেয়ী ও অলি-আউলিয়া"
7447,"নবী-রাসূল, সাহাবা, তাবেয়ী ও অলি-আউলিয়া"
27768,"নবী-রাসূল, সাহাবা, তাবেয়ী ও অলি-আউলিয়া"
27770,"নবী-রাসূল, সাহাবা, তাবেয়ী ও অলি-আউলিয়া"
60915,"নবী-রাসূল, সাহাবা, তাবেয়ী ও অলি-আউলিয়া"
60921,"নবী-রাসূল, সাহাবা, তাবেয়ী ও অলি-আউলিয়া"
60923,"নবী-রাসূল, সাহাবা, তাবেয়ী ও অলি-আউলিয়া"
60925,"নবী-রাসূল, সাহাবা, তাবেয়ী ও অলি-আউলিয়া"
60926,"নবী-রাসূল, সাহাবা, তাবেয়ী ও অলি-আউলিয়া"
60927,"নবী-রাসূল, সাহাবা, তাবেয়ী ও অলি-আউলিয়া"


In [36]:
df_bn.loc[df_bn['categories'] == 'নবী-রাসূল, সাহাবা, তাবেয়ী ও অলি-আউলিয়া', 'categories'] = "নবী-রাসুল, সাহাবী, তাবেঈ, ওলি-আউলিয়া"

In [30]:
df_bn.loc[df_bn['language_categories'] == 'fr', 'categories'] = "যখন ৪-৮, উপকথা"

In [31]:
df_bn.loc[df_bn['language_categories'] == 'vi', 'categories'] = "টোফেল"

In [32]:
df_bn.loc[df_bn['language_categories'] == 'en', 'categories'] = "উক্তি, বাণী, প্রবাদ"

### 3.2 Stripping unnecessary characters

In [37]:
df_bn.loc[:, 'categories_fixed'] = df_bn['categories'].apply(remove_unnecessary_characters)

In [38]:
df_bn.loc[:, 'categories_fixed']

,categories_fixed
0,পশ্চিমবঙ্গের বই
1,"পশ্চিমবঙ্গের বই, সমকালীন গল্প"
2,"পশ্চিমবঙ্গের বই, শিশু-কিশোর উপন্যাস"
3,ব্যাংকিং এন্ড কমার্স ল
4,"অ্যাডভোকেসি, বিচার আইন"
...,...
204567,"পশ্চিমবঙ্গের বই, জীবনী, চিঠিপত্র, ডায়েরি, স্মৃ..."
204568,বাংলা-ইংরেজি কবিতা
204569,বাংলা-ইংরেজি কবিতা
204570,"অতিপ্রাকৃত, ভৌতিক"


### 3.2.1 Keeping only the bangla characters

In [39]:
df_bn.loc[:, 'categories_fixed'] = df_bn['categories_fixed'].apply(keep_only_bangla)

In [40]:
# sanity check

df_bn.loc[df_bn['categories'] == ""]

,book_id,title,author,publisher,publisher_name_english,categories,category_english,edition,isbn,summary,...,offer_price,book_url,prod_img_link,availability,product_category,actual_rating,bangla_title,language_title,language_categories,categories_fixed


In [41]:
df_bn.loc[:, 'categories_fixed']

,categories_fixed
0,পশ্চিমবঙ্গের বই
1,"পশ্চিমবঙ্গের বই, সমকালীন গল্প"
2,"পশ্চিমবঙ্গের বই, শিশু-কিশোর উপন্যাস"
3,ব্যাংকিং এন্ড কমার্স ল
4,"অ্যাডভোকেসি, বিচার আইন"
...,...
204567,"পশ্চিমবঙ্গের বই, জীবনী, চিঠিপত্র, ডায়েরি, স্মৃ..."
204568,বাংলা-ইংরেজি কবিতা
204569,বাংলা-ইংরেজি কবিতা
204570,"অতিপ্রাকৃত, ভৌতিক"


In [42]:
for cat in df_bn.loc[:, 'categories_fixed'].unique():
    print(cat)

পশ্চিমবঙ্গের বই
পশ্চিমবঙ্গের বই, সমকালীন গল্প
পশ্চিমবঙ্গের বই, শিশু-কিশোর উপন্যাস
ব্যাংকিং এন্ড কমার্স ল
অ্যাডভোকেসি, বিচার আইন
প্রোপার্টি ল
ম্যানুয়ালস লস
ডায়েরি, চিঠি, স্মৃতি
চতুর্থ শ্রেণি
ঈমান, আক্বিদা, তাওবাহ
প্রথম শ্রেণি
কিন্ডারগার্টেন, নার্সারি
কিন্ডারগার্টেন, প্লে গ্রুপ
স্মারকগ্রন্থ, বিবিধ
সাহিত্যিক, শিল্প, সংগীত ব্যক্তিত্ব
স্বাস্থ্যবিধি, পরামর্শ
বাংলা কবিতা
সমকালীন উপন্যাস
শিশু, কিশোর কালেকশন
বিবিধ বিষয়ক প্রবন্ধ
গল্প, শিশু, কিশোর
বয়স যখন ০-৪, শিশুতোষ
ভর্তি, নিয়োগ, প্রস্তুতি পরীক্ষা, বিবিধ
জীবনী, স্মৃতিচারণ, বিবিধ
রহস্য, গোয়েন্দা, ভৌতিক, মিথ, থ্রিলার, অ্যাডভেঞ্চার, অনুবাদ, ইংরেজি
শিশু-কিশোর গল্প
ফটোগ্রাফি কৌশল
ছোটদের গণিত, বিজ্ঞান, প্রযুক্তি
কবিতা সমগ্র
চিরায়ত উপন্যাস
পশ্চিমবঙ্গের বই, প্রবন্ধ
পশ্চিমবঙ্গের বই, বাংলা কবিতা
অনুবাদ, আত্ম-উন্নয়ন, মেডিটেশন
পশ্চিমবঙ্গের বই, ভাষা, অভিধান
ইতিহাস, ঐতিহ্য, গবেষণা, প্রবন্ধ
কলকাতা পুস্তকমেলা ২০২৪
পশ্চিমবঙ্গের বই, সংগীত, চলচ্চিত্র, ফটোগ্রাফি
পশ্চিমবঙ্গের বই, সাহিত্য সমালোচনা
পশ্চিমবঙ্গের বই, সাহিত্যিক, শিল্প, সংগীত ব্যক্তিত্ব
পশ্চিমবঙ্গের বই

In [43]:
df_bn.loc[df_bn['categories_fixed'] == "ফ্যামিলি , পার্সোনাল লস", 'categories_fixed']

,categories_fixed
72237,"ফ্যামিলি , পার্সোনাল লস"
116571,"ফ্যামিলি , পার্সোনাল লস"


In [44]:
# নারী, শিশু ল
# ফ্যামিলি , পার্সোনাল লস
# অন্যরা
df_bn.loc[df_bn['categories_fixed'] == "ফ্যামিলি , পার্সোনাল লস", 'categories_fixed'] = "ফ্যামিলি, পার্সোনাল লস"

In [ ]:
# text = "ক্লাস ৯, ১০ এসএসসি, রসায়ন পাঠ সোহায়িকা, পাঠো সোহায়িকা"
text = "ক্লাস ৯-১০, বিজ্ঞান, ইংরেজি পাঠ সোহায়িকা"
text = re.sub(r'ক্লাস ৯, ১০', "ক্লাস ৯-১০,", text)
text = re.sub(r"(পাঠ\s)সোহায়িকা|(পাঠো\s)সোহায়িকা", "পাঠ্য সহায়িকা", text)
# text = re.sub(r"(পাঠো\s)সোহায়িকা", "পাঠ্য সহায়িকা", text)
print(text)

ক্লাস ৯-১০, বিজ্ঞান, ইংরেজি পাঠ্য সহায়িকা


In [46]:
df_bn.loc[:, 'categories_fixed'] = df_bn['categories_fixed'].apply(fix_more_categories_issue)

In [47]:
# no more
df_bn.loc[df_bn['categories_fixed'].str.contains("সোহায়িকা")]

,book_id,title,author,publisher,publisher_name_english,categories,category_english,edition,isbn,summary,...,offer_price,book_url,prod_img_link,availability,product_category,actual_rating,bangla_title,language_title,language_categories,categories_fixed


In [48]:
# পাঠো
df_bn.loc[df_bn['categories_fixed'].str.contains("ক্লাস ৯, ১০")]

,book_id,title,author,publisher,publisher_name_english,categories,category_english,edition,isbn,summary,...,offer_price,book_url,prod_img_link,availability,product_category,actual_rating,bangla_title,language_title,language_categories,categories_fixed


In [49]:
df_bn.to_csv("rokomari_books_only_bangla_v2.csv", index=False)

# Rough khata

In [ ]:
category_counts_df['cat_1'].value_counts()

,count
cat_1,
পশ্চিমবঙ্গের বই,98
এইচএসসি ১ম বর্ষ,42
এইচএসসি ২য় বর্ষ,37
শিশু-কিশোর,32
বয়স যখন ৮-১২,28
...,...
"ভর্তি, নিয়োগ ও প্রস্তুতি পরীক্ষা",1
বয়স যখন ৪-৮,1
ইতিহাসে বাংলাদেশ,1


In [ ]:
category_1 = category_counts_df['cat_1'].value_counts()

In [ ]:
category_1[5:41]

,count
cat_1,
বয়স যখন ১২-১৭,26
When 4-8,14
বয়স যখন ৪-৮,13
অনুবাদ,12
মুক্তিযুদ্ধের ইতিহাস,11
বয়স যখন ৪-৮,11
When 8-12,10
ব্যবস্থাপনা বিভাগ,10
উদ্ভিদবিজ্ঞান বিভাগ,9


In [ ]:
category_counts_df.loc[category_counts_df['cat_1'] == "Foreign Language Books"]

,categories,count,cat_1,cat_2,cat_3,categories_split,category_length
362,Foreign Language Books: Other Foreign Books,6,Foreign Language Books,Other Foreign Books,None,"[Foreign Language Books, Other Foreign Books]",2
558,Foreign Language Books: Islamic Books,2,Foreign Language Books,Islamic Books,None,"[Foreign Language Books, Islamic Books]",2
627,Foreign Language Books: Religious Books,1,Foreign Language Books,Religious Books,None,"[Foreign Language Books, Religious Books]",2
680,"Foreign Language Books: Mathematics, Science &...",1,Foreign Language Books,"Mathematics, Science & Technology",None,"[Foreign Language Books, Mathematics, Science...",2
683,Foreign Language Books: Self-help and meditation,1,Foreign Language Books,Self-help and meditation,None,"[Foreign Language Books, Self-help and medita...",2
719,"Foreign Language Books: Professional, Journal ...",1,Foreign Language Books,"Professional, Journal and Reference",None,"[Foreign Language Books, Professional, Journa...",2


In [ ]:
df_bn.loc[df_bn['categories'].str.contains(r'^Foreign Language Books:', regex=True)]

,author,categories,isbn,n_ratings,n_reviews,price,prod_img_link,publisher,rating,summary,title,language,summary_availability,language_author,language_categories
20496,মুনসি প্রেমচাঁদ,Foreign Language Books: Other Foreign Books,9798187335282,1,No Review,360,No image,Nai Sadi Book House,3.0,No summary,গোডান (হিন্দি),bn,No,bn,en
29632,এ আর কে শর্মা,Foreign Language Books: Other Foreign Books,9789383606115,1,No Review,180,https://ds.rokomari.store/rokomari110/ProductN...,Sri Sarada Book House (India),5.0,No summary,স্ট্রেন্থ ফার্স্ট গুডনেস নেক্সট,bn,No,bn,en
31080,পাওলো কোয়েলহো,Foreign Language Books: Other Foreign Books,9788183280921,No Ratings,No Reviews,405,No image,Wisdom Tree (India),0.0,No summary,জাহির (হিন্দি),bn,No,bn,en
33418,ড ভি আবদুর রহিম,"Foreign Language Books: Professional, Journal ...",No ISBN,2,No Review,7600,No image,U.K. Islamic Academy,3.0,No summary,এরাবিক কোর্স ফর ইংলিশ স্পিকিং স্টুডেন্টস (৩ভলি...,bn,No,bn,en
62024,বি কে শিবানী,Foreign Language Books: Other Foreign Books,9788182748576,1,No Review,351,https://ds.rokomari.store/rokomari110/ProductN...,Manjul Publishing House Pvt. Ltd (India),2.0,No summary,আসেম আনন্দ কি আর (হ্যাপিনেস আনলিমিটেড),bn,No,bn,en
62028,রোন্ডা বাইর্ন,Foreign Language Books: Other Foreign Books,9788183220941,1,No Review,718,https://ds.rokomari.store/rokomari110/ProductN...,Manjul Publishing House Pvt. Ltd (India),5.0,No summary,দ্যা সিক্রেট)(হিন্দি),bn,No,bn,en
66124,খান স্যার পাটনা,Foreign Language Books: Other Foreign Books,No ISBN,2,No Review,225,https://ds.rokomari.store/rokomari110/ProductN...,Kiran Prakashan,5.0,No summary,এনসিইআরটি ৩০০০ ফ্যাক্টস,bn,No,bn,en
76148,শায়খ সফিউর রহমান মোবারকপুরী,Foreign Language Books: Islamic Books,9786035000710,No Ratings,No Reviews,3960,No image,Darussalam (India),0.0,No summary,কালেকশন ফ্রম রিয়াদ উস সালেহীন,bn,No,bn,en
76156,দেভ আল কারদায়ী,Foreign Language Books: Religious Books,No ISBN,No Ratings,No Reviews,2800,No image,Darussalam (India),0.0,No summary,ডিকশনারি অব ইসলামিক টার্ম,bn,No,bn,en
94196,প্রসন্ত কুমার গিরি,"Foreign Language Books: Mathematics, Science &...",9789387162051,No Ratings,No Reviews,765,No image,Academic Publishers (Indian),0.0,No summary,ইন্ট্রোডাকশন টু স্ট্যাটিসটিক্স ইনক্লুডিং স্ট্য...,bn,No,bn,en


In [ ]:
print(category_1[category_1 == 1])

cat_1
কম্পিউটার সায়েন্স অ্যান্ড টেকনোলজি                      1
এইচএসসি মানবিক বিভাগ                                    1
Poem and Rhyme                                          1
Others                                                  1
Children & Teen Magazine                                1
বইমেলা ২০১৩                                             1
Self-Development                                        1
এনভায়রনমেন্টাল ইঞ্জিনিয়ারিং                             1
রহস্য, গোয়েন্দা, ভৌতিক, মিথ, থ্রিলার, ও অ্যাডভেঞ্চার    1
Islamic Book                                            1
রকমারি নির্বাচিত বই                                     1
বিদেশী ফটোগ্রাফি ম্যাগাজিন                              1
History                                                 1
Self-Help, Motivational and Meditation                  1
HSC and Equivalent                                      1
টেস্ট পেপারস                                            1
Social and Political Magazine                           1
HSC Sugg

In [ ]:
category_counts_df['cat_2'].value_counts()

,count
cat_2,
অনার্স ১ম বর্ষ,24
অনার্স ৩য় বর্ষ,22
অনার্স ২য় বর্ষ,22
অনার্স ৪র্থ বর্ষ,20
ডিগ্রি ১ম বর্ষ,19
...,...
ইন্টারভিউ,1
জীববিজ্ঞান পাঠ্য সহায়িকা,1
মানবিক বিভাগ পাঠ্য সহায়িকা,1


In [ ]:
category_2 = category_counts_df['cat_2'].value_counts()
print(category_2[category_2 == 1])

cat_2
 Science Department Pattho Sohayika             1
 রান্না, স্বাস্থ্য, ফ্যাশন ও পরিবার বিষয়ক বই    1
আইসিটি পাঠ্য সহায়িকা                           1
 হিসাববিজ্ঞান পাঠ্য সহায়িকা                    1
 এসএসসি ও সমমান                                 1
                                               ..
 ইন্টারভিউ                                      1
 জীববিজ্ঞান পাঠ্য সহায়িকা                      1
 মানবিক বিভাগ পাঠ্য সহায়িকা                    1
 বিবিধ প্রসঙ্গ                                  1
 ইংরেজি পাঠ্য সহায়িকা                          1
Name: count, Length: 262, dtype: int64


In [ ]:
# save the modified dataframe

# df_bn.to_csv("rokomari_books_categories_corrected.csv", index=False)

In [ ]:
# df_bn.loc[df_bn['language_categories'] == 'en']

In [ ]:
# category_counts_df = pd.read_csv("/content/drive/MyDrive/Rokomari Recommendation Dataset/category_counts.csv")

In [ ]:
# def filter_cat(text):
#     return ':' in text

# category_counts_df = df_bn[df_bn['categories'].apply(filter_cat)]['categories'].value_counts().reset_index()

# another way to do this
category_counts_df = df_bn.loc[df_bn['categories'].str.contains(":"), 'categories'].value_counts().reset_index()

In [ ]:
# this is not necessary, also it's a good practice not to
# capitalize names

# category_counts_df.columns = ['Category', 'Count']
category_counts_df

,categories,count
0,পশ্চিমবঙ্গের বই: প্রবন্ধ,924
1,পশ্চিমবঙ্গের বই: বাংলা কবিতা,694
2,অনুবাদ: আত্ম-উন্নয়ন ও মেডিটেশন,617
3,"শিশু-কিশোর: রহস্য, গোয়েন্দা, ভৌতিক, থ্রিলার ও ...",521
4,পশ্চিমবঙ্গের বই: রচনাসমগ্র ও সংকলন,515
...,...,...
800,নবম ও দশম (বিজ্ঞান): ধর্ম ও নৈতিক শিক্ষা পাঠ্য...,1
801,ইঞ্জিনিয়ারিং: পরীক্ষার প্রস্তুতি,1
802,মুক্তিযুদ্ধের ইতিহাস: বিবিধ,1
803,নবম ও দশম (ব্যবসায় শিক্ষা): বাংলাদেশ ও বিশ্ব প...,1


In [ ]:
# category_counts_df.to_csv('category_counts.csv', index=False)

In [ ]:
# df_bn['language_categories'] = df_bn['categories'].apply(detect_language)

In [ ]:
# df_bn['language_categories'].value_counts()

In [ ]:
df_bn.loc[df_bn['language_categories'].isin(["no", "sv", "pt", "vi", "cy"])]

,author,categories,isbn,n_ratings,n_reviews,price,prod_img_link,publisher,rating,summary,title,language,summary_availability,language_author,language_categories
1010,কবি মমতাজ রেনু,Poem Criticism,9843226488,No Ratings,No Reviews,430,https://ds.rokomari.store/rokomari110/ProductN...,Ahsania Books,0.00,"Ilhami(Inspired) Poetess Mumtaz Renu, a profou...",এঞ্জেল অন মাই শোল্ডারস্,bn,Yes,bn,pt
1012,কবি মমতাজ রেনু,Poem Criticism,No ISBN,No Ratings,No Reviews,1720,https://ds.rokomari.store/rokomari110/ProductN...,Ahsania Books,0.00,No summary,মেরি সুরাত তেরা আয়না,bn,No,bn,pt
9953,এসেনশিয়াল পাবলিকেশন সম্পাদক,TOEFL,No ISBN,7,7 Reviews,226,https://ds.rokomari.store/rokomari110/ProductN...,এসেনশিয়াল পাবলিকেশন,4.86,No summary,Cliffs Toefl Grand Review বাংলা ভার্সন,bn,No,bn,vi
9954,এসেনশিয়াল পাবলিকেশন সম্পাদক,TOEFL,No ISBN,No Ratings,No Reviews,226,https://ds.rokomari.store/rokomari110/ProductN...,এসেনশিয়াল পাবলিকেশন,0.00,CONTENTS,Cliffs Toefl Grand Review বাংলা ভার্সন,bn,Yes,bn,vi
11416,সোনিয়া পারভিন,Cyber Law,No ISBN,No Ratings,No Reviews,174,No image,Hasan Law Books,0.00,No summary,"সাইবার নিরাপত্তা আইন, ২০২৩",bn,No,bn,cy
14331,জয়কলি সম্পাদনা পরিষদ সম্পাদক,TOEFL,No ISBN,No Ratings,No Reviews,213,No image,জয়কলি পাবলিকেশন্স লিঃ,0.00,"সকল প্রশ্নের নির্ভুল উত্তর, সঠিক ব্যাখ্যা, প্র...",বারণ’স এন্ড ক্লিফস টোয়েফল - বাংলা অনুবাদ,bn,Yes,bn,vi
24467,মেজর শের মোহাম্মদ খান অবঃ অনুবাদক,Rhymes,978-984-97262-7-2,No Ratings,No Reviews,129,https://ds.rokomari.store/rokomari110/ProductN...,Raowa Publication,0.00,ছোটো সোনামনিদের ছড়ার বই। এই বইয়ের প্রতিটি ছড়া ...,ছােটােদর বাংলা ছড়া ও English Rhymes,bn,Yes,bn,cy
33028,বিচারপতি মোঃ আজিজুল হক,Cyber Law,9789849156499,No Ratings,No Reviews,898,https://ds.rokomari.store/rokomari110/ProductN...,Universal Book House,0.00,No summary,সাইবার ল এন্ড ক্রাইম,bn,No,bn,cy
33138,অ্যাডভোকেট জাহাঙ্গীর আলম সরকার,Cyber Law,978984892811,No Ratings,No Reviews,627,No image,Unique Law Book House,0.00,"১. সাইবার নিরাপত্তা আইন,২০২৩",সাইবার নিরাপত্তা আইনের ভাষ্য ও পর্নোগ্রাফি নিয়...,bn,Yes,bn,cy
34443,প্রফেসর মোঃ হাফিজুর রহমান,Honors,No ISBN,No Ratings,No Reviews,264,No image,Titas Publications,0.00,No summary,লিনিয়ার এলজাবরা - অনার্স ৩য় বর্ষ,bn,No,bn,no


In [ ]:
df_bn.loc[df_bn['language_categories'].isin(["no", "sv", "pt", "vi", "cy"])].shape

(25, 15)

In [ ]:
category_counts_df = category_counts_df.join(
    category_counts_df['categories'].str.split(':', expand=True).rename(
        columns={0: 'cat_1', 1: 'cat_2', 2: 'cat_3'}
    )
)

In [ ]:
category_counts_df['categories_split'] = category_counts_df['categories'].str.split(":")

In [ ]:
category_counts_df

,categories,count,cat_1,cat_2,cat_3,categories_split
0,পশ্চিমবঙ্গের বই: প্রবন্ধ,924,পশ্চিমবঙ্গের বই,প্রবন্ধ,None,"[পশ্চিমবঙ্গের বই, প্রবন্ধ]"
1,পশ্চিমবঙ্গের বই: বাংলা কবিতা,694,পশ্চিমবঙ্গের বই,বাংলা কবিতা,None,"[পশ্চিমবঙ্গের বই, বাংলা কবিতা]"
2,অনুবাদ: আত্ম-উন্নয়ন ও মেডিটেশন,617,অনুবাদ,আত্ম-উন্নয়ন ও মেডিটেশন,None,"[অনুবাদ, আত্ম-উন্নয়ন ও মেডিটেশন]"
3,"শিশু-কিশোর: রহস্য, গোয়েন্দা, ভৌতিক, থ্রিলার ও ...",521,শিশু-কিশোর,"রহস্য, গোয়েন্দা, ভৌতিক, থ্রিলার ও অ্যাডভেঞ্চার",None,"[শিশু-কিশোর, রহস্য, গোয়েন্দা, ভৌতিক, থ্রিলার ..."
4,পশ্চিমবঙ্গের বই: রচনাসমগ্র ও সংকলন,515,পশ্চিমবঙ্গের বই,রচনাসমগ্র ও সংকলন,None,"[পশ্চিমবঙ্গের বই, রচনাসমগ্র ও সংকলন]"
...,...,...,...,...,...,...
800,নবম ও দশম (বিজ্ঞান): ধর্ম ও নৈতিক শিক্ষা পাঠ্য...,1,নবম ও দশম (বিজ্ঞান),ধর্ম ও নৈতিক শিক্ষা পাঠ্য সহায়িকা,None,"[নবম ও দশম (বিজ্ঞান), ধর্ম ও নৈতিক শিক্ষা পাঠ..."
801,ইঞ্জিনিয়ারিং: পরীক্ষার প্রস্তুতি,1,ইঞ্জিনিয়ারিং,পরীক্ষার প্রস্তুতি,None,"[ইঞ্জিনিয়ারিং, পরীক্ষার প্রস্তুতি]"
802,মুক্তিযুদ্ধের ইতিহাস: বিবিধ,1,মুক্তিযুদ্ধের ইতিহাস,বিবিধ,None,"[মুক্তিযুদ্ধের ইতিহাস, বিবিধ]"
803,নবম ও দশম (ব্যবসায় শিক্ষা): বাংলাদেশ ও বিশ্ব প...,1,নবম ও দশম (ব্যবসায় শিক্ষা),বাংলাদেশ ও বিশ্ব পরিচয় পাঠ্য সহায়িকা,None,"[নবম ও দশম (ব্যবসায় শিক্ষা), বাংলাদেশ ও বিশ্ব..."


In [ ]:
category_counts_df['category_length'] = category_counts_df['categories_split'].apply(lambda x: len(x))

In [ ]:
category_counts_df.head()

,categories,count,cat_1,cat_2,cat_3,categories_split,category_length
0,পশ্চিমবঙ্গের বই: প্রবন্ধ,924,পশ্চিমবঙ্গের বই,প্রবন্ধ,None,"[পশ্চিমবঙ্গের বই, প্রবন্ধ]",2
1,পশ্চিমবঙ্গের বই: বাংলা কবিতা,694,পশ্চিমবঙ্গের বই,বাংলা কবিতা,None,"[পশ্চিমবঙ্গের বই, বাংলা কবিতা]",2
2,অনুবাদ: আত্ম-উন্নয়ন ও মেডিটেশন,617,অনুবাদ,আত্ম-উন্নয়ন ও মেডিটেশন,None,"[অনুবাদ, আত্ম-উন্নয়ন ও মেডিটেশন]",2
3,"শিশু-কিশোর: রহস্য, গোয়েন্দা, ভৌতিক, থ্রিলার ও ...",521,শিশু-কিশোর,"রহস্য, গোয়েন্দা, ভৌতিক, থ্রিলার ও অ্যাডভেঞ্চার",None,"[শিশু-কিশোর, রহস্য, গোয়েন্দা, ভৌতিক, থ্রিলার ...",2
4,পশ্চিমবঙ্গের বই: রচনাসমগ্র ও সংকলন,515,পশ্চিমবঙ্গের বই,রচনাসমগ্র ও সংকলন,None,"[পশ্চিমবঙ্গের বই, রচনাসমগ্র ও সংকলন]",2


In [ ]:
category_counts_df['category_length'].value_counts()

,count
category_length,
2,802
3,3


In [ ]:
# to get the values where cat_3 == None

category_counts_df.loc[(category_counts_df['cat_3'].isna())]

,categories,count,cat_1,cat_2,cat_3,categories_split,category_length
0,পশ্চিমবঙ্গের বই: প্রবন্ধ,924,পশ্চিমবঙ্গের বই,প্রবন্ধ,None,"[পশ্চিমবঙ্গের বই, প্রবন্ধ]",2
1,পশ্চিমবঙ্গের বই: বাংলা কবিতা,694,পশ্চিমবঙ্গের বই,বাংলা কবিতা,None,"[পশ্চিমবঙ্গের বই, বাংলা কবিতা]",2
2,অনুবাদ: আত্ম-উন্নয়ন ও মেডিটেশন,617,অনুবাদ,আত্ম-উন্নয়ন ও মেডিটেশন,None,"[অনুবাদ, আত্ম-উন্নয়ন ও মেডিটেশন]",2
3,"শিশু-কিশোর: রহস্য, গোয়েন্দা, ভৌতিক, থ্রিলার ও ...",521,শিশু-কিশোর,"রহস্য, গোয়েন্দা, ভৌতিক, থ্রিলার ও অ্যাডভেঞ্চার",None,"[শিশু-কিশোর, রহস্য, গোয়েন্দা, ভৌতিক, থ্রিলার ...",2
4,পশ্চিমবঙ্গের বই: রচনাসমগ্র ও সংকলন,515,পশ্চিমবঙ্গের বই,রচনাসমগ্র ও সংকলন,None,"[পশ্চিমবঙ্গের বই, রচনাসমগ্র ও সংকলন]",2
...,...,...,...,...,...,...,...
800,নবম ও দশম (বিজ্ঞান): ধর্ম ও নৈতিক শিক্ষা পাঠ্য...,1,নবম ও দশম (বিজ্ঞান),ধর্ম ও নৈতিক শিক্ষা পাঠ্য সহায়িকা,None,"[নবম ও দশম (বিজ্ঞান), ধর্ম ও নৈতিক শিক্ষা পাঠ...",2
801,ইঞ্জিনিয়ারিং: পরীক্ষার প্রস্তুতি,1,ইঞ্জিনিয়ারিং,পরীক্ষার প্রস্তুতি,None,"[ইঞ্জিনিয়ারিং, পরীক্ষার প্রস্তুতি]",2
802,মুক্তিযুদ্ধের ইতিহাস: বিবিধ,1,মুক্তিযুদ্ধের ইতিহাস,বিবিধ,None,"[মুক্তিযুদ্ধের ইতিহাস, বিবিধ]",2
803,নবম ও দশম (ব্যবসায় শিক্ষা): বাংলাদেশ ও বিশ্ব প...,1,নবম ও দশম (ব্যবসায় শিক্ষা),বাংলাদেশ ও বিশ্ব পরিচয় পাঠ্য সহায়িকা,None,"[নবম ও দশম (ব্যবসায় শিক্ষা), বাংলাদেশ ও বিশ্ব...",2


`Fill the `None` values with empty ("") string`

In [ ]:
category_counts_df.loc[:, 'cat_3'].fillna("", inplace=True)

In [ ]:
category_counts_df.head()

,categories,count,cat_1,cat_2,cat_3,categories_split,category_length
0,পশ্চিমবঙ্গের বই: প্রবন্ধ,924,পশ্চিমবঙ্গের বই,প্রবন্ধ,,"[পশ্চিমবঙ্গের বই, প্রবন্ধ]",2
1,পশ্চিমবঙ্গের বই: বাংলা কবিতা,694,পশ্চিমবঙ্গের বই,বাংলা কবিতা,,"[পশ্চিমবঙ্গের বই, বাংলা কবিতা]",2
2,অনুবাদ: আত্ম-উন্নয়ন ও মেডিটেশন,617,অনুবাদ,আত্ম-উন্নয়ন ও মেডিটেশন,,"[অনুবাদ, আত্ম-উন্নয়ন ও মেডিটেশন]",2
3,"শিশু-কিশোর: রহস্য, গোয়েন্দা, ভৌতিক, থ্রিলার ও ...",521,শিশু-কিশোর,"রহস্য, গোয়েন্দা, ভৌতিক, থ্রিলার ও অ্যাডভেঞ্চার",,"[শিশু-কিশোর, রহস্য, গোয়েন্দা, ভৌতিক, থ্রিলার ...",2
4,পশ্চিমবঙ্গের বই: রচনাসমগ্র ও সংকলন,515,পশ্চিমবঙ্গের বই,রচনাসমগ্র ও সংকলন,,"[পশ্চিমবঙ্গের বই, রচনাসমগ্র ও সংকলন]",2


In [ ]:
category_counts_df.tail()

,categories,count,cat_1,cat_2,cat_3,categories_split,category_length
800,নবম ও দশম (বিজ্ঞান): ধর্ম ও নৈতিক শিক্ষা পাঠ্য...,1,নবম ও দশম (বিজ্ঞান),ধর্ম ও নৈতিক শিক্ষা পাঠ্য সহায়িকা,,"[নবম ও দশম (বিজ্ঞান), ধর্ম ও নৈতিক শিক্ষা পাঠ...",2
801,ইঞ্জিনিয়ারিং: পরীক্ষার প্রস্তুতি,1,ইঞ্জিনিয়ারিং,পরীক্ষার প্রস্তুতি,,"[ইঞ্জিনিয়ারিং, পরীক্ষার প্রস্তুতি]",2
802,মুক্তিযুদ্ধের ইতিহাস: বিবিধ,1,মুক্তিযুদ্ধের ইতিহাস,বিবিধ,,"[মুক্তিযুদ্ধের ইতিহাস, বিবিধ]",2
803,নবম ও দশম (ব্যবসায় শিক্ষা): বাংলাদেশ ও বিশ্ব প...,1,নবম ও দশম (ব্যবসায় শিক্ষা),বাংলাদেশ ও বিশ্ব পরিচয় পাঠ্য সহায়িকা,,"[নবম ও দশম (ব্যবসায় শিক্ষা), বাংলাদেশ ও বিশ্ব...",2
804,ইসলামের ইতিহাস ও সংস্কৃতি বিভাগ: অনার্স ৪র্থ বর্ষ,1,ইসলামের ইতিহাস ও সংস্কৃতি বিভাগ,অনার্স ৪র্থ বর্ষ,,"[ইসলামের ইতিহাস ও সংস্কৃতি বিভাগ, অনার্স ৪র্থ...",2


In [ ]:
category_counts_df.loc[category_counts_df['cat_1'] == "শিশু-কিশোর"]

,categories,count,cat_1,cat_2,cat_3,categories_split,category_length
3,"শিশু-কিশোর: রহস্য, গোয়েন্দা, ভৌতিক, থ্রিলার ও ...",521,শিশু-কিশোর,"রহস্য, গোয়েন্দা, ভৌতিক, থ্রিলার ও অ্যাডভেঞ্চার",,"[শিশু-কিশোর, রহস্য, গোয়েন্দা, ভৌতিক, থ্রিলার ...",2
22,শিশু-কিশোর: বিবিধ,212,শিশু-কিশোর,বিবিধ,,"[শিশু-কিশোর, বিবিধ]",2
32,শিশু-কিশোর: বিখ্যাত ব্যক্তি ও জীবনী,148,শিশু-কিশোর,বিখ্যাত ব্যক্তি ও জীবনী,,"[শিশু-কিশোর, বিখ্যাত ব্যক্তি ও জীবনী]",2
45,"শিশু-কিশোর: রূপকথা, উপকথা ও লোককাহিনী",107,শিশু-কিশোর,"রূপকথা, উপকথা ও লোককাহিনী",,"[শিশু-কিশোর, রূপকথা, উপকথা ও লোককাহিনী]",2
50,শিশু-কিশোর: রেফারেন্স ও আত্মউন্নয়নমূলক বই,95,শিশু-কিশোর,রেফারেন্স ও আত্মউন্নয়নমূলক বই,,"[শিশু-কিশোর, রেফারেন্স ও আত্মউন্নয়নমূলক বই]",2
67,শিশু-কিশোর: রচনাসমগ্র/সংকলন,70,শিশু-কিশোর,রচনাসমগ্র/সংকলন,,"[শিশু-কিশোর, রচনাসমগ্র/সংকলন]",2
68,শিশু-কিশোর: ছবি আঁকা ও গান,69,শিশু-কিশোর,ছবি আঁকা ও গান,,"[শিশু-কিশোর, ছবি আঁকা ও গান]",2
69,শিশু-কিশোর: সাধারণ জ্ঞান,67,শিশু-কিশোর,সাধারণ জ্ঞান,,"[শিশু-কিশোর, সাধারণ জ্ঞান]",2
79,"শিশু-কিশোর: গণিত, বিজ্ঞান ও প্রযুক্তি",57,শিশু-কিশোর,"গণিত, বিজ্ঞান ও প্রযুক্তি",,"[শিশু-কিশোর, গণিত, বিজ্ঞান ও প্রযুক্তি]",2
84,শিশু-কিশোর: ভাষা আন্দোলন ও মুক্তিযুদ্ধ,54,শিশু-কিশোর,ভাষা আন্দোলন ও মুক্তিযুদ্ধ,,"[শিশু-কিশোর, ভাষা আন্দোলন ও মুক্তিযুদ্ধ]",2


In [ ]:
category_counts_df.loc[category_counts_df['cat_1'] == "অনুবাদ"]

,categories,count,cat_1,cat_2,cat_3,categories_split,category_length
2,অনুবাদ: আত্ম-উন্নয়ন ও মেডিটেশন,617,অনুবাদ,আত্ম-উন্নয়ন ও মেডিটেশন,,"[অনুবাদ, আত্ম-উন্নয়ন ও মেডিটেশন]",2
498,"অনুবাদ: জীবনী, স্মৃতিচারণ ও সাক্ষাৎকার",3,অনুবাদ,"জীবনী, স্মৃতিচারণ ও সাক্ষাৎকার",,"[অনুবাদ, জীবনী, স্মৃতিচারণ ও সাক্ষাৎকার]",2
567,অনুবাদ: প্রবন্ধ,2,অনুবাদ,প্রবন্ধ,,"[অনুবাদ, প্রবন্ধ]",2
593,অনুবাদ: শিশু-কিশোর,2,অনুবাদ,শিশু-কিশোর,,"[অনুবাদ, শিশু-কিশোর]",2
629,অনুবাদ: কমিকস ও নকশা,1,অনুবাদ,কমিকস ও নকশা,,"[অনুবাদ, কমিকস ও নকশা]",2
646,অনুবাদ: ইসলাম,1,অনুবাদ,ইসলাম,,"[অনুবাদ, ইসলাম]",2
657,অনুবাদ: বিবিধ ধর্মীয় বই,1,অনুবাদ,বিবিধ ধর্মীয় বই,,"[অনুবাদ, বিবিধ ধর্মীয় বই]",2
660,অনুবাদ: পৌরাণিক কাহিনী,1,অনুবাদ,পৌরাণিক কাহিনী,,"[অনুবাদ, পৌরাণিক কাহিনী]",2
686,"অনুবাদ: ব্যবসা, বিনিয়োগ ও অর্থনীতি",1,অনুবাদ,"ব্যবসা, বিনিয়োগ ও অর্থনীতি",,"[অনুবাদ, ব্যবসা, বিনিয়োগ ও অর্থনীতি]",2
687,"অনুবাদ:রহস্য, গোয়েন্দা, ভৌতিক, মিথ, থ্রিলার, ও...",1,অনুবাদ,"রহস্য, গোয়েন্দা, ভৌতিক, মিথ, থ্রিলার, ও অ্যাডভ...",,"[অনুবাদ, রহস্য, গোয়েন্দা, ভৌতিক, মিথ, থ্রিলার,...",2


In [ ]:
category_counts_df.loc[category_counts_df['cat_2'].str.contains("পাঠ্য সহায়িকা", regex=True)]

,categories,count,cat_1,cat_2,cat_3,categories_split,category_length
100,এইচএসসি ১ম বর্ষ: গণিত পাঠ্য সহায়িকা,43,এইচএসসি ১ম বর্ষ,গণিত পাঠ্য সহায়িকা,,"[এইচএসসি ১ম বর্ষ, গণিত পাঠ্য সহায়িকা]",2
111,এইচএসসি ১ম বর্ষ: পদার্থবিজ্ঞান পাঠ্য সহায়িকা,38,এইচএসসি ১ম বর্ষ,পদার্থবিজ্ঞান পাঠ্য সহায়িকা,,"[এইচএসসি ১ম বর্ষ, পদার্থবিজ্ঞান পাঠ্য সহায়িকা]",2
123,এইচএসসি ১ম বর্ষ: রসায়ন পাঠ্য সহায়িকা,33,এইচএসসি ১ম বর্ষ,রসায়ন পাঠ্য সহায়িকা,,"[এইচএসসি ১ম বর্ষ, রসায়ন পাঠ্য সহায়িকা]",2
180,এইচএসসি ১ম বর্ষ: জীববিজ্ঞান পাঠ্য সহায়িকা,21,এইচএসসি ১ম বর্ষ,জীববিজ্ঞান পাঠ্য সহায়িকা,,"[এইচএসসি ১ম বর্ষ, জীববিজ্ঞান পাঠ্য সহায়িকা]",2
230,এইচএসসি ২য় বর্ষ: পদার্থবিজ্ঞান পাঠ্য সহায়িকা,16,এইচএসসি ২য় বর্ষ,পদার্থবিজ্ঞান পাঠ্য সহায়িকা,,"[এইচএসসি ২য় বর্ষ, পদার্থবিজ্ঞান পাঠ্য সহায়িকা]",2
284,এইচএসসি ২য় বর্ষ: রসায়ন পাঠ্য সহায়িকা,11,এইচএসসি ২য় বর্ষ,রসায়ন পাঠ্য সহায়িকা,,"[এইচএসসি ২য় বর্ষ, রসায়ন পাঠ্য সহায়িকা]",2
313,এইচএসসি ২য় বর্ষ: গণিত পাঠ্য সহায়িকা,9,এইচএসসি ২য় বর্ষ,গণিত পাঠ্য সহায়িকা,,"[এইচএসসি ২য় বর্ষ, গণিত পাঠ্য সহায়িকা]",2
381,এইচএসসি ২য় বর্ষ: জীববিজ্ঞান পাঠ্য সহায়িকা,6,এইচএসসি ২য় বর্ষ,জীববিজ্ঞান পাঠ্য সহায়িকা,,"[এইচএসসি ২য় বর্ষ, জীববিজ্ঞান পাঠ্য সহায়িকা]",2
389,এইচএসসি ১ম বর্ষ: তথ্য ও যোগাযোগ প্রযুক্তি পাঠ্...,5,এইচএসসি ১ম বর্ষ,তথ্য ও যোগাযোগ প্রযুক্তি পাঠ্য সহায়িকা,,"[এইচএসসি ১ম বর্ষ, তথ্য ও যোগাযোগ প্রযুক্তি পা...",2
455,এইচএসসি ২য় বর্ষ: সমাজবিজ্ঞান/সমাজকর্ম পাঠ্য সহ...,4,এইচএসসি ২য় বর্ষ,সমাজবিজ্ঞান/সমাজকর্ম পাঠ্য সহায়িকা,,"[এইচএসসি ২য় বর্ষ, সমাজবিজ্ঞান/সমাজকর্ম পাঠ্য ...",2


In [ ]:
category_counts_df.loc[category_counts_df['cat_2'].str.contains("অনার্স", regex=True)]

,categories,count,cat_1,cat_2,cat_3,categories_split,category_length
87,রাষ্ট্রবিজ্ঞান বিভাগ: অনার্স ১ম বর্ষ,52,রাষ্ট্রবিজ্ঞান বিভাগ,অনার্স ১ম বর্ষ,,"[রাষ্ট্রবিজ্ঞান বিভাগ, অনার্স ১ম বর্ষ]",2
104,ভূগোল বিভাগ: অনার্স ১ম বর্ষ,41,ভূগোল বিভাগ,অনার্স ১ম বর্ষ,,"[ভূগোল বিভাগ, অনার্স ১ম বর্ষ]",2
110,রাষ্ট্রবিজ্ঞান বিভাগ: অনার্স ৪র্থ বর্ষ,38,রাষ্ট্রবিজ্ঞান বিভাগ,অনার্স ৪র্থ বর্ষ,,"[রাষ্ট্রবিজ্ঞান বিভাগ, অনার্স ৪র্থ বর্ষ]",2
117,ইতিহাস বিভাগ: অনার্স ১ম বর্ষ,36,ইতিহাস বিভাগ,অনার্স ১ম বর্ষ,,"[ইতিহাস বিভাগ, অনার্স ১ম বর্ষ]",2
118,রাষ্ট্রবিজ্ঞান বিভাগ: অনার্স ২য় বর্ষ,35,রাষ্ট্রবিজ্ঞান বিভাগ,অনার্স ২য় বর্ষ,,"[রাষ্ট্রবিজ্ঞান বিভাগ, অনার্স ২য় বর্ষ]",2
...,...,...,...,...,...,...,...
753,অর্থনীতি বিভাগ: অনার্স ৩য় বর্ষ,1,অর্থনীতি বিভাগ,অনার্স ৩য় বর্ষ,,"[অর্থনীতি বিভাগ, অনার্স ৩য় বর্ষ]",2
755,পরিসংখ্যান বিভাগ: অনার্স ২য় বর্ষ,1,পরিসংখ্যান বিভাগ,অনার্স ২য় বর্ষ,,"[পরিসংখ্যান বিভাগ, অনার্স ২য় বর্ষ]",2
767,লোক প্রশাসন বিভাগ: অনার্স ৩য় বর্ষ,1,লোক প্রশাসন বিভাগ,অনার্স ৩য় বর্ষ,,"[লোক প্রশাসন বিভাগ, অনার্স ৩য় বর্ষ]",2
772,ইসলামের ইতিহাস ও সংস্কৃতি বিভাগ: অনার্স ৩য় বর্ষ,1,ইসলামের ইতিহাস ও সংস্কৃতি বিভাগ,অনার্স ৩য় বর্ষ,,"[ইসলামের ইতিহাস ও সংস্কৃতি বিভাগ, অনার্স ৩য় ব...",2


In [ ]:
category_counts_df.loc[category_counts_df['cat_1'].str.contains("নবম ও দশম", regex=True)]

,categories,count,cat_1,cat_2,cat_3,categories_split,category_length
122,নবম ও দশম (এসএসসি): কমন সাবজেক্ট,34,নবম ও দশম (এসএসসি),কমন সাবজেক্ট,,"[নবম ও দশম (এসএসসি), কমন সাবজেক্ট]",2
135,নবম ও দশম (এসএসসি): আবশ্যিক বিষয়,28,নবম ও দশম (এসএসসি),আবশ্যিক বিষয়,,"[নবম ও দশম (এসএসসি), আবশ্যিক বিষয়]",2
149,নবম ও দশম (বিজ্ঞান): পদার্থবিজ্ঞান পাঠ্য সহায়িকা,26,নবম ও দশম (বিজ্ঞান),পদার্থবিজ্ঞান পাঠ্য সহায়িকা,,"[নবম ও দশম (বিজ্ঞান), পদার্থবিজ্ঞান পাঠ্য সহা...",2
162,নবম ও দশম (বিজ্ঞান): রসায়ন পাঠ্য সহায়িকা,23,নবম ও দশম (বিজ্ঞান),রসায়ন পাঠ্য সহায়িকা,,"[নবম ও দশম (বিজ্ঞান), রসায়ন পাঠ্য সহায়িকা]",2
207,নবম ও দশম (এসএসসি): বিজ্ঞান বিভাগ,19,নবম ও দশম (এসএসসি),বিজ্ঞান বিভাগ,,"[নবম ও দশম (এসএসসি), বিজ্ঞান বিভাগ]",2
243,নবম ও দশম (এসএসসি): ব্যবসায় শিক্ষা বিভাগ,14,নবম ও দশম (এসএসসি),ব্যবসায় শিক্ষা বিভাগ,,"[নবম ও দশম (এসএসসি), ব্যবসায় শিক্ষা বিভাগ]",2
248,নবম ও দশম (এসএসসি): বিজ্ঞান বিভাগ পাঠ্য সহায়িকা,13,নবম ও দশম (এসএসসি),বিজ্ঞান বিভাগ পাঠ্য সহায়িকা,,"[নবম ও দশম (এসএসসি), বিজ্ঞান বিভাগ পাঠ্য সহায...",2
273,নবম ও দশম (এসএসসি): মানবিক বিভাগ পাঠ্য সহায়িকা,11,নবম ও দশম (এসএসসি),মানবিক বিভাগ পাঠ্য সহায়িকা,,"[নবম ও দশম (এসএসসি), মানবিক বিভাগ পাঠ্য সহায়...",2
279,নবম ও দশম (বিজ্ঞান): জীববিজ্ঞান পাঠ্য সহায়িকা,11,নবম ও দশম (বিজ্ঞান),জীববিজ্ঞান পাঠ্য সহায়িকা,,"[নবম ও দশম (বিজ্ঞান), জীববিজ্ঞান পাঠ্য সহায়িকা]",2
336,নবম ও দশম (এসএসসি): ব্যবসায় শিক্ষা বিভাগ পাঠ্...,8,নবম ও দশম (এসএসসি),ব্যবসায় শিক্ষা বিভাগ পাঠ্য সহায়িকা,,"[নবম ও দশম (এসএসসি), ব্যবসায় শিক্ষা বিভাগ পা...",2
